In [ ]:
"""
run_pipeline.py - runs the production steps in order and stops at the first failure.
Customer Health & Sentiment Scoring Multi-Agent AI Solution

    python run_pipeline.py                 # 2 -> 3a -> 3b -> 3c -> 3d
    python run_pipeline.py --from 3a       # Bronze already exists (skip ingestion)
    python run_pipeline.py --from 3c       # resume from feature engineering
    python run_pipeline.py --only 3d       # rebuild the cohort only
    python run_pipeline.py --lenient       # 3c: warn (do not stop) on business-mapping mismatches

The EDA (eda_data_understanding_v6.py) is NOT part of this run: it is an analysis notebook that informs
business_rules.py (mappings, ranges, CORE_RULE). Run it after Step 2 and before freezing business_rules.py.

Human gate: after the first 3a run, review config/allowed_values_suggested.json and save it as
config/allowed_values.json. 3b runs without it but then skips allowed-value checks.

Change log (see decisions.md): D03 Step 2 added to the runner, D08 a crashing step is reported as FAILED with the
resume command instead of an unhandled traceback, D09 runs from a notebook too.
"""

## 1. Imports and step registry

In [ ]:
from __future__ import annotations

import argparse
import time
import traceback

import step2_data_ingestion
import step3a_profile
import step3b_validate_clean
import step3c_feature_engineering
import step3d_build_cohort
from common import ALLOWED_VALUES_FILE, finish, in_notebook, log

STEPS = ["2", "3a", "3b", "3c", "3d"]

## 2. Run one step

In [ ]:
def run(step: str, lenient: bool) -> bool:
    if step == "2":
        return step2_data_ingestion.main()
    if step == "3a":
        return step3a_profile.main()
    if step == "3b":
        if not ALLOWED_VALUES_FILE.exists():
            log("WARNING: config/allowed_values.json missing - 3b will skip allowed-value checks")
        return step3b_validate_clean.main()
    if step == "3c":
        return step3c_feature_engineering.main(strict=not lenient)
    if step == "3d":
        return step3d_build_cohort.main()
    raise ValueError(step)

## 3. Orchestration

In [ ]:
def main(argv: list[str] | None = None) -> bool:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--from", dest="start", choices=STEPS, default=STEPS[0])
    p.add_argument("--only", choices=STEPS)
    p.add_argument("--lenient", action="store_true")
    a = p.parse_args(argv)
    steps = [a.only] if a.only else STEPS[STEPS.index(a.start):]

    for s in steps:
        t0 = time.time()
        log(f"========== STEP {s} ==========")
        try:
            ok = run(s, a.lenient)
        except Exception:                       # report, do not crash the runner
            traceback.print_exc()
            ok = False
        log(f"STEP {s} {'PASSED' if ok else 'FAILED'} in {time.time() - t0:,.1f}s")
        if not ok:
            log(f"Pipeline stopped at {s}. Fix the reported checks and resume with --from {s}.")
            return False
    log("Pipeline completed.")
    return True

In [ ]:
if __name__ == "__main__":
    # In a notebook edit the list, e.g. ["--from", "3a"]; from a terminal the command-line arguments are used.
    finish(main(["--from", "3a"] if in_notebook() else None))